In [6]:
def run_shared_task_inference(input_csv_path, output_csv_path):
    # 1. Load data
    df = pd.read_csv(input_csv_path)
    ensemble = load_ensemble()
    all_model_logits = []

    # 2. Run each model in the ensemble
    for name, components in ensemble.items():
        print(f"Processing {name}...")
        model = components["model"]
        tokenizer = components["tokenizer"]
        
        # Ensure make_preprocess_fn is available in your namespace
        prep_fn = make_preprocess_fn(tokenizer) 
        ds = Dataset.from_pandas(df).map(prep_fn, batched=False)
        ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'logic_features'])
        
        dataloader = torch.utils.data.DataLoader(ds, batch_size=16)
        model_logits = []
        
        with torch.no_grad():
            for batch in dataloader:
                batch = {k: v.to(device) for k, v in batch.items()}
                outputs = model(**batch)
                model_logits.append(outputs["logits"].cpu().numpy())
        
        all_model_logits.append(np.concatenate(model_logits, axis=0))

    # 3. Ensemble Averaging (Soft Voting)
    avg_logits = np.mean(all_model_logits, axis=0)
    final_preds = np.argmax(avg_logits, axis=1)
    
    # 4. FORMAT FOR SHARED TASK
    # Create a new DataFrame with ONLY the required column
    submission_df = pd.DataFrame({'prediction': final_preds})
    
    # 5. Save without the index
    submission_df.to_csv(output_csv_path, index=False)
    
    print(f"\n--- Predictions Format for Shared Task ---")
    print(f"File saved to: {output_csv_path}")
    print(f"Total predictions: {len(submission_df)}")
    print(submission_df.head()) # Preview the first few rows

# --- EXECUTION ---
# Change "test_data.csv" to your actual test filename
run_shared_task_inference("training_data/NLI/dev.csv", "shared_task_predictions.csv")

Loading deberta from nli_ensemble_model2/deberta...


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

RuntimeError: Error(s) in loading state_dict for POSSpecializedMoE:
	Missing key(s) in state_dict: "encoder.encoder.layer.6.attention.self.query_proj.weight", "encoder.encoder.layer.6.attention.self.query_proj.bias", "encoder.encoder.layer.6.attention.self.key_proj.weight", "encoder.encoder.layer.6.attention.self.key_proj.bias", "encoder.encoder.layer.6.attention.self.value_proj.weight", "encoder.encoder.layer.6.attention.self.value_proj.bias", "encoder.encoder.layer.6.attention.output.dense.weight", "encoder.encoder.layer.6.attention.output.dense.bias", "encoder.encoder.layer.6.attention.output.LayerNorm.weight", "encoder.encoder.layer.6.attention.output.LayerNorm.bias", "encoder.encoder.layer.6.intermediate.dense.weight", "encoder.encoder.layer.6.intermediate.dense.bias", "encoder.encoder.layer.6.output.dense.weight", "encoder.encoder.layer.6.output.dense.bias", "encoder.encoder.layer.6.output.LayerNorm.weight", "encoder.encoder.layer.6.output.LayerNorm.bias", "encoder.encoder.layer.

In [1]:
import os
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import spacy
from transformers import AutoModel, AutoTokenizer, AutoConfig, DefaultDataCollator
from datasets import Dataset

# --- 1. PREPROCESSING LOGIC (Pasted here to avoid ImportErrors) ---
try:
    nlp = spacy.load("en_core_web_sm")
except:
    import os
    os.system("python -m spacy download en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

def get_pos_filtered_text(text, pos_tags):
    doc = nlp(str(text))
    tokens = [token.text for token in doc if token.pos_ in pos_tags]
    return " ".join(tokens) if tokens else "none"

def extract_logic_features(premise, hypothesis):
    p_doc, h_doc = nlp(str(premise)), nlp(str(hypothesis))
    neg_p = sum(1 for t in p_doc if t.dep_ == "neg")
    neg_h = sum(1 for t in h_doc if t.dep_ == "neg")
    p_set = {t.lemma_.lower() for t in p_doc if not t.is_stop and not t.is_punct}
    h_set = {t.lemma_.lower() for t in h_doc if not t.is_stop and not t.is_punct}
    overlap = len(p_set & h_set) / len(p_set | h_set) if (p_set | h_set) else 0.0
    return [float(neg_p), float(neg_h), float(abs(neg_p - neg_h)), float(overlap)]

def make_preprocess_fn(tokenizer):
    def preprocess(example):
        main_enc = tokenizer(example["premise"], example["hypothesis"], truncation=True, padding="max_length", max_length=128)
        p_nouns = get_pos_filtered_text(example["premise"], ["NOUN", "PROPN"])
        h_nouns = get_pos_filtered_text(example["hypothesis"], ["NOUN", "PROPN"])
        p_verbs = get_pos_filtered_text(example["premise"], ["VERB"])
        h_verbs = get_pos_filtered_text(example["hypothesis"], ["VERB"])
        
        ent_enc = tokenizer(p_nouns, h_nouns, truncation=True, padding="max_length", max_length=128)
        act_enc = tokenizer(p_verbs, h_verbs, truncation=True, padding="max_length", max_length=128)
        
        return {
            "input_ids": main_enc["input_ids"],
            "attention_mask": main_enc["attention_mask"],
            "entity_ids": ent_enc["input_ids"],
            "action_ids": act_enc["input_ids"],
            "logic_features": extract_logic_features(example["premise"], example["hypothesis"])
        }
    return preprocess

# --- 2. MODEL ARCHITECTURE ---
class POSSpecializedMoE(nn.Module):
    def __init__(self, model_name, num_labels=2):
        super().__init__()
        config = AutoConfig.from_pretrained(model_name)
        self.encoder = AutoModel.from_pretrained(model_name, config=config).float() 
        hidden_size = self.encoder.config.hidden_size
        
        self.semantic_expert = nn.Linear(hidden_size, hidden_size)
        self.entity_expert = nn.Linear(hidden_size, hidden_size)
        self.action_expert = nn.Linear(hidden_size, hidden_size)
        self.logic_expert = nn.Linear(4, hidden_size)
        
        self.logic_norm = nn.LayerNorm(4)
        self.gating = nn.Sequential(
            nn.Linear(hidden_size + 4, 128),
            nn.ReLU(),
            nn.Linear(128, 4),
            nn.Softmax(dim=-1)
        )
        self.classifier = nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids, attention_mask, entity_ids=None, action_ids=None, logic_features=None, **kwargs):
        # Base encoding
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :].float() 

        # Experts
        e_sem = torch.tanh(self.semantic_expert(cls_output))
        
        # Entity expert (uses specialized input)
        e_ent_hidden = self.encoder(input_ids=entity_ids, attention_mask=attention_mask).last_hidden_state[:, 0, :]
        e_ent = torch.tanh(self.entity_expert(e_ent_hidden))
        
        # Action expert (uses specialized input)
        e_act_hidden = self.encoder(input_ids=action_ids, attention_mask=attention_mask).last_hidden_state[:, 0, :]
        e_act = torch.tanh(self.action_expert(e_act_hidden))
        
        # Logic expert
        norm_logic = self.logic_norm(logic_features.float())
        e_log = torch.tanh(self.logic_expert(norm_logic))

        # Gating logic
        gate_input = torch.cat([cls_output, norm_logic], dim=-1)
        gate_weights = self.gating(gate_input) 

        experts = torch.stack([e_sem, e_ent, e_act, e_log], dim=1)
        moe_output = torch.bmm(gate_weights.unsqueeze(1), experts).squeeze(1)

        return {"logits": self.classifier(moe_output)}

# --- 3. INFERENCE ENGINE ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def run_shared_task_inference(input_csv_path, output_csv_path):
    ensemble_path = "nli_ensemble_model2"
    backbones = {
        "deberta": "microsoft/deberta-v3-small",
        "modernbert": "answerdotai/ModernBERT-base"
    }
    
    df = pd.read_csv(input_csv_path)
    all_model_logits = []

    for name, path in backbones.items():
        print(f"Loading and Running {name}...")
        specific_dir = os.path.join(ensemble_path, name)
        
        tokenizer = AutoTokenizer.from_pretrained(specific_dir)
        model = POSSpecializedMoE(path)
        model.load_state_dict(torch.load(os.path.join(specific_dir, "moe_weights.pt"), map_location=device))
        model.to(device).eval()
        
        prep_fn = make_preprocess_fn(tokenizer)
        ds = Dataset.from_pandas(df).map(prep_fn, batched=False)
        ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'entity_ids', 'action_ids', 'logic_features'])
        
        loader = torch.utils.data.DataLoader(ds, batch_size=16, collate_fn=DefaultDataCollator())
        
        logits = []
        with torch.no_grad():
            for batch in loader:
                batch = {k: v.to(device) for k, v in batch.items()}
                logits.append(model(**batch)["logits"].cpu().numpy())
        
        all_model_logits.append(np.concatenate(logits, axis=0))

    final_logits = np.mean(all_model_logits, axis=0)
    df['prediction'] = np.argmax(final_logits, axis=1)
    df[['prediction']].to_csv(output_csv_path, index=False)
    print(f"Submission saved to {output_csv_path}")

# Run it!
run_shared_task_inference("training_data/NLI/dev.csv", "final_submission.csv")

Loading and Running deberta...


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Map:   0%|          | 0/6736 [00:00<?, ? examples/s]

Loading and Running modernbert...


Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
decoder.bias      | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Map:   0%|          | 0/6736 [00:00<?, ? examples/s]

Submission saved to final_submission.csv
